# Carve3D full reconstruction + MRC pipeline

This notebook runs a complete, open-checkpoint replacement for the components not released by Carve3D: four real views → LGM Gaussian reconstruction → same-camera renders → LPIPS MRC.

Before running, enable **Internet** and a **GPU accelerator** in Kaggle. Put four images in a Kaggle Dataset and name them `view_000`, `view_090`, `view_180`, and `view_270` (png/jpg/jpeg/webp). They must show the same centered object at azimuths 0°, 90°, 180°, 270°, with roughly the same elevation.

In [ ]:
# Clone this branch after it has been pushed.
REPO_URL = 'https://github.com/huytrao/carve3d-test.git'
REPO_DIR = '/kaggle/working/carve3d-test'
LGM_DIR = '/kaggle/working/LGM'
!git clone --branch implement_full_pipeline $REPO_URL $REPO_DIR
%cd $REPO_DIR
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!bash scripts/setup_lgm_kaggle.sh $LGM_DIR

In [ ]:
# Change only this path to your Kaggle Dataset folder.
INPUT_DIR = '/kaggle/input/YOUR-FOUR-VIEWS-DATASET'
OUTPUT_DIR = '/kaggle/working/carve3d-output'
!find $INPUT_DIR -maxdepth 1 -type f | sort
!python full_pipeline/run.py --lgm-root $LGM_DIR --input-dir $INPUT_DIR --output-dir $OUTPUT_DIR

In [ ]:
# Inspect the numerical MRC result (lower is better) and visual diagnostics.
import json
from IPython.display import Image, Video, display

with open(f'{OUTPUT_DIR}/metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics['mrc'], indent=2))
display(Image(filename=f'{OUTPUT_DIR}/input_grid.png'))
display(Image(filename=f'{OUTPUT_DIR}/render_grid.png'))
display(Video(f'{OUTPUT_DIR}/orbit.mp4', embed=True))
print('PLY 3D asset:', f'{OUTPUT_DIR}/reconstruction.ply')

## Optional prompt route

This route uses MVDream's public checkpoint to generate four input views, then follows the same LGM/MRC pipeline. Select Kaggle **T4 x2**: GPU 0 samples MVDream and GPU 1 reconstructs/renders with LGM. It is a runnable baseline, not the unpublished Carve3D RL-finetuned checkpoint.

In [ ]:
PROMPT_OUTPUT = '/kaggle/working/carve3d-prompt-output'
!python kaggle_full_pipeline_prompt.py --lgm-root $LGM_DIR --prompt 'a wooden chair' --seed 42 --output-dir $PROMPT_OUTPUT